# Complete Unsloth Guide (2026, Official Patterns)

This guide is distilled from the **Unsloth team's official Colab notebooks**. Not made up — these are the real patterns.

Reference notebooks I drew from:
- `Gemma4_(E2B)_Text.ipynb`
- `Qwen3_(4B)_Instruct.ipynb`
- `Qwen3_(4B)_Thinking.ipynb`
- `Qwen3_5_(2B)_Vision.ipynb`

## Table of Contents

1. **The Class Family** — which class for which job?
2. **The `unsloth.chat_templates` Module** — the heart of the official approach
3. **Text SFT Pattern** — with `train_on_responses_only`
4. **Vision SFT Pattern** — with `UnslothVisionDataCollator`
5. **Thinking Model SFT** — reasoning data + `enable_thinking`
6. **Hyperparameter Guide** — official defaults
7. **Inference Patterns**
8. **Save & Deploy**
9. **Error Catalog**
10. **Decision Flow + Quick Start**

## 1. The Class Family — Which One When?

What the official notebooks show:

| Class | Model Type | Official Notebook That Uses It |
|-------|-----------|--------------------------|
| **`FastLanguageModel`** | Text-only LLM | Qwen3 4B Instruct/Thinking |
| **`FastModel`** | Text-only LLM (newer unified API) | Gemma 4 E2B Text |
| **`FastVisionModel`** | Vision-Language Model | Qwen3.5 2B Vision |

**Important:** `FastLanguageModel` is NOT legacy — the official Qwen3 notebook still uses it. `FastModel` is the newer unified API but both are active.

**Practical rule:**
- For text SFT: `FastLanguageModel` or `FastModel` (either works; pick by model)
- For vision SFT: **`FastVisionModel`** (mandatory)
- Mixed (a VLM you're using text-only): `FastModel` is the safer pick

## Decision Flow

```
What kind of model is it?
│
├── Text-only LLM (Qwen3, Llama-3, Mistral)
│   ├── Following the official Qwen3 notebook → FastLanguageModel
│   └── Want the modern unified API → FastModel
│
├── VLM, text-only fine-tune (Gemma 4 E2B text task)
│   └── FastModel + finetune_vision_layers=False
│
└── VLM, vision SFT (Qwen3.5 2B Vision)
    └── FastVisionModel + UnslothVisionDataCollator
```

In [ ]:
import unsloth
[c for c in dir(unsloth) if 'Fast' in c or 'Model' in c]

## 2. The `unsloth.chat_templates` Module — Heart of the Official Approach

Every official notebook uses these 3 functions:

```python
from unsloth.chat_templates import (
    get_chat_template,           # apply chat template to the tokenizer
    standardize_data_formats,    # normalize dataset format
    train_on_responses_only,     # completion-only loss mask
)
```

### 2.1 `get_chat_template`

Applies a model-specific chat template to the tokenizer:

```python
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-thinking")
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
```

Built-in template names (partial list):
- `gemma-4`, `gemma-3`, `gemma-2`, `gemma-1`
- `qwen3-instruct`, `qwen3-thinking`, `qwen-2.5`
- `llama-3`, `llama-3.1`, `llama-3.2`
- `mistral`, `phi-4`

### 2.2 `standardize_data_formats`

Converts various dataset formats (sharegpt, alpaca, etc.) into the standard `conversations` format:

```python
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split="train")
dataset = standardize_data_formats(dataset)
# Now there's a 'conversations' column and the format is normalized
```

### 2.3 `train_on_responses_only` — CRITICAL

This is **separate from TRL's `assistant_only_loss=True`**. It's Unsloth's own approach and is decoupled from the VLM check.

```python
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(...)  # build the trainer first

# Then wrap — mask the instruction so only the response contributes to loss
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",      # for Qwen3
    response_part = "<|im_start|>assistant\n",    # for Qwen3
)
```

### Instruction/Response Tokens — Model-Specific

| Model Family | `instruction_part` | `response_part` |
|--------------|---------------------|-----------------|
| **Qwen3** (Instruct/Thinking) | `<|im_start|>user\n` | `<|im_start|>assistant\n` |
| **Gemma 4** | `<|turn>user\n` | `<|turn>model\n` |
| **Llama 3** | `<|start_header_id|>user<|end_header_id|>\n\n` | `<|start_header_id|>assistant<|end_header_id|>\n\n` |
| **Mistral** | `[INST]` | `[/INST]` |

You can read the tokens straight from the tokenizer:
```python
sample_text = tokenizer.apply_chat_template(messages, tokenize=False)
print(sample_text[:200])  # inspect the format
```

## 3. Text SFT Pattern — Official Flow

Based on the official `Qwen3_(4B)_Instruct.ipynb`:

In [ ]:
import unsloth                                              # 1. very first import
from unsloth import FastLanguageModel
from unsloth.chat_templates import (
    get_chat_template, standardize_data_formats, train_on_responses_only,
)
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,
    load_in_4bit = True,                  # QLoRA
    load_in_8bit = False,
    full_finetuning = False,
)

# 3. LoRA — classic target_modules list (common in the official notebooks)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,                                # 8/16/32/64/128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,                       # typically same as r
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state = 3407,
    use_rslora = False,                    # RSLoRA is optional
    loftq_config = None,                   # LoftQ is optional
)

# 4. Chat template
tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

# 5. Dataset — standardize + format
dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:1000]")
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# 6. Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,                 # tokenizer=, not processing_class= — official notebooks still pass tokenizer=
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,                   # fixed step count (not warmup_ratio)
        max_steps = 60,                     # or num_train_epochs = 1
        learning_rate = 2e-4,               # for LoRA
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,               # 0.001 — official default
        lr_scheduler_type = "linear",       # linear — official default
        seed = 3407,
        report_to = "none",
    ),
)

# 7. CRITICAL — masking via train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# 8. Train
trainer_stats = trainer.train()

### Verify That Masking Works

The official notebooks always include this check:

```python
# Sample 100 — full input
print(tokenizer.decode(trainer.train_dataset[100]["input_ids"]))
# Shows instruction + response

# Sample 100 — only the labeled (unmasked) part
print(tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in trainer.train_dataset[100]["labels"]
]))
# Shows only the response; the instruction is filled with `<pad>`
```

Expected: only the assistant response is visible; the user message is replaced by `<pad>` tokens.

### Same Pattern for Gemma 4 (different details)

Based on the official `Gemma4_(E2B)_Text.ipynb`:

```python
from unsloth import FastModel              # not FastLanguageModel!

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,                           # auto detection
    max_seq_length = 1024,
    load_in_4bit = False,                   # False for 16-bit LoRA
    full_finetuning = False,
)

# LoRA — finetune_* flags + r=8 for Gemma 4
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,    # off for text-only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,                                  # small rank
    lora_alpha = 8,                         # alpha == r
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

# Chat template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

# ... formatting + trainer ...

# train_on_responses_only with Gemma 4 tokens
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)
```

## 4. Vision SFT Pattern

Based on the official `Qwen3_5_(2B)_Vision.ipynb`. **Completely different pattern** — no `train_on_responses_only`, but `UnslothVisionDataCollator` instead.

In [ ]:
import unsloth
from unsloth import FastVisionModel              # separate class
from unsloth.trainer import UnslothVisionDataCollator
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 1. Model — FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit = False,                  # 16-bit LoRA
    use_gradient_checkpointing = "unsloth",
)

# 2. LoRA — finetune_* flags (vision included)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,     # True for vision SFT
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    # target_modules = "all-linear",        # optional
)

# 3. Vision data — image + text content
dataset = load_dataset("unsloth/LaTeX_OCR", split="train")

instruction = "Write the LaTeX representation for this image."

def convert_to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": sample["image"]},   # PIL.Image
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["text"]},
            ]},
        ]
    }
converted_dataset = [convert_to_conversation(s) for s in dataset]

# 4. Training — CRITICAL config
FastVisionModel.for_training(model)        # switch to training mode

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),  # MANDATORY
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        # MANDATORY trio for vision SFT:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

trainer_stats = trainer.train()

## 5. Thinking Model SFT (Reasoning)

Based on the official `Qwen3_(4B)_Thinking.ipynb`. Same as the text SFT pattern — only the data and chat template differ:

```python
# Model — Thinking variant
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Chat template — thinking variant
tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-thinking")

# Dataset — reasoning data (math/CoT)
dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")

def generate_conversation(examples):
    return {"conversations": [
        [
            {"role": "user", "content": problem},
            {"role": "assistant", "content": solution},
        ]
        for problem, solution in zip(examples["problem"], examples["generated_solution"])
    ]}
dataset = dataset.map(generate_conversation, batched=True)
```

**Inference is different — the `enable_thinking` parameter:**

```python
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,                  # True/False — turn the reasoning blocks on/off
)
```

In training, every example has thinking blocks (CoT). At inference time, `enable_thinking=False` lets you get just the final answer.

## 6. Hyperparameter Guide (Official Defaults)

### LoRA Parameters

| Param | Official Value | Note |
|-------|-------------|-----|
| `r` | 8 (Gemma E2B), 16, 32 (Qwen3 4B) | Pick by model size |
| `lora_alpha` | typically equal to r | r=8 → α=8, r=32 → α=32 |
| `lora_dropout` | 0 | Optimized for speed |
| `bias` | `"none"` | Optimized |
| `use_gradient_checkpointing` | `"unsloth"` | 30% less VRAM |
| `random_state` | 3407 | Reproducibility |
| `use_rslora` | False | RSLoRA is supported (consider it at rank ≥ 64) |
| `loftq_config` | None | LoftQ is supported |

### Two Ways To Specify LoRA Targets

**Method 1 — classic `target_modules`** (used in the Qwen3 notebooks):
```python
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
```

**Method 2 — modern `finetune_*`** (used in the Gemma 4 and Vision notebooks):
```python
finetune_vision_layers     = False,
finetune_language_layers   = True,
finetune_attention_modules = True,
finetune_mlp_modules       = True,
```

**Method 3 — single string `"all-linear"`:**
```python
target_modules = "all-linear"   # optional, mentioned in the vision notebook
```

These usually produce the same result. The `finetune_*` flags are **preferred for VLMs** (so you can split vision vs language).

### Training Parameters (SFTConfig)

| Param | Official Default | Notes |
|-------|---------------|--------|
| `per_device_train_batch_size` | 2 | Increase based on GPU |
| `gradient_accumulation_steps` | 4 | Effective = batch × accum |
| `warmup_steps` | **5** | Fixed step count, not a ratio |
| `max_steps` | 30-60 (demo) | Production: `num_train_epochs=1` |
| `learning_rate` | **2e-4** | For LoRA. Long training: 2e-5 |
| `logging_steps` | 1 | Log every step |
| `optim` | **`"adamw_8bit"`** | bitsandbytes 8-bit |
| `weight_decay` | **0.001** | (I had said 0.01 before — wrong) |
| `lr_scheduler_type` | **`"linear"`** | (Not cosine!) |
| `seed` | 3407 | |
| `report_to` | `"none"` | Override for wandb / trackio |

### Vision SFT Extra Parameters

```python
remove_unused_columns = False
dataset_text_field = ""
dataset_kwargs = {"skip_prepare_dataset": True}
max_length = 2048
```

**Without these three, vision data is not parsed.**

## 7. Inference Patterns

### Standard Text Inference

```python
from transformers import TextStreamer

messages = [{"role": "user", "content": "What is interest?"}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,            # MANDATORY for generation
)

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens = 1000,
    streamer = TextStreamer(tokenizer, skip_prompt=True),
    # Generation params vary by model!
)
```

### Generation Params — Model-Specific Recommendations

| Model | temperature | top_p | top_k | min_p |
|-------|-------------|-------|-------|-------|
| Qwen3 (non-thinking) | 0.7 | 0.8 | 20 | — |
| Qwen3 thinking mode | different | — | — | — |
| Gemma 4 | 1.0 | 0.95 | 64 | — |
| Qwen3.5 Vision | 1.5 | — | — | 0.1 |

### Thinking Inference

```python
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,                 # turn reasoning off
)
```

### Vision Inference

```python
messages = [{
    "role": "user",
    "content": [
        {"type": "image"},                   # placeholder
        {"type": "text", "text": "Describe this image."},
    ]
}]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

# Image goes as the 1st arg to the tokenizer:
inputs = tokenizer(
    image,                                    # PIL.Image first
    input_text,                              # text second
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature = 1.5, min_p = 0.1,
)
```

## 8. Save & Deploy

The official notebooks use 4 save methods:

### A. LoRA Adapter
```python
model.save_pretrained("qwen_lora")
tokenizer.save_pretrained("qwen_lora")
```

### B. Push to Hub (LoRA only)
```python
model.push_to_hub("HF_ACCOUNT/qwen_lora", token="hf_xxx")
tokenizer.push_to_hub("HF_ACCOUNT/qwen_lora", token="hf_xxx")
```

### C. Merged 16-bit (for vLLM)
```python
model.save_pretrained_merged(
    "qwen_merged",
    tokenizer,
    save_method = "merged_16bit",
)
```

### D. GGUF (Ollama / llama.cpp)
```python
# Single quant
model.save_pretrained_gguf(
    "qwen_gguf",
    tokenizer,
    quantization_method = "q4_k_m",     # or q8_0, f16
)

# Multiple quants at once
model.push_to_hub_gguf(
    "HF_ACCOUNT/qwen_gguf",
    tokenizer,
    quantization_method = ["q4_k_m", "q8_0", "f16"],
    token = "hf_xxx",
)
```

### Loading the LoRA (in a new session)
```python
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "qwen_lora",            # saved folder or HF repo
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
```

## 9. Error Catalog

### Error 1: `Unsloth: Should be imported before trl, transformers, peft`
**Cause:** wrong import order.
**Fix:** put `import unsloth` at the very top.

### Error 2: `KeyError: 'qwen3_5'`
**Cause:** transformers is too old.
**Fix:** install `transformers>=5.5`.

### Error 3: `libcusparseLt.so.0` or `libnvshmem_host.so.3`
**Cause:** missing CUDA dependency after a lock upgrade.
**Fix:** `uv pip install --reinstall nvidia-cusparselt-cu12 nvidia-nvshmem-cu12 torch`

### Error 4: `flash-attn` warning
**Cause:** flash-attn doesn't match your Python version.
**Fix:** ignore it — Xformers fallback, ~10% speed loss.

### Error 5: `RuntimeError: ... formatting_func`
**Cause:** got conversational format but `formatting_prompts_func` wasn't written.
**Fix:** follow the official pattern — build the text field with `dataset.map(formatting_prompts_func)`.

### Error 6: All labels = -100 in vision SFT
**Cause:** you used `assistant_only_loss=True` (incompatible with VLMs).
**Fix:** use `UnslothVisionDataCollator` + the mandatory triple config; don't set `assistant_only_loss`.

### Error 7: KeyError on dataset fields in vision SFT
**Cause:** `remove_unused_columns=True` (default) — drops vision columns.
**Fix:** set `remove_unused_columns=False`.

### Error 8: Out of VRAM
**Try, in order:**
1. `load_in_4bit=True` (QLoRA)
2. Increase `gradient_accumulation_steps`, decrease `per_device_batch_size`
3. `use_gradient_checkpointing="unsloth"`
4. Decrease `max_seq_length`
5. Decrease `r`

## 10. Quick Start — Official Pattern Template

The template below is 95% faithful to the official **Qwen3 4B Instruct** notebook. You can swap the model and the dataset.

In [ ]:
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import (
    get_chat_template, standardize_data_formats, train_on_responses_only,
)
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# ============================================================
# 1. Model
# ============================================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)

# ============================================================
# 2. LoRA
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# ============================================================
# 3. Chat template + Data
# ============================================================
tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:1000]")
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    return {
        "text": [
            tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
            for c in convos
        ]
    }
dataset = dataset.map(formatting_prompts_func, batched=True)

# ============================================================
# 4. Trainer
# ============================================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,                      # demo; production: num_train_epochs=1
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

# ============================================================
# 5. Train
# ============================================================
trainer_stats = trainer.train()

# ============================================================
# 6. Inference
# ============================================================
from transformers import TextStreamer
messages = [{"role": "user", "content": "What is interest?"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens = 500,
    temperature = 0.7, top_p = 0.8, top_k = 20,
    streamer = TextStreamer(tokenizer, skip_prompt=True),
)

# ============================================================
# 7. Save
# ============================================================
model.save_pretrained("qwen_lora")
tokenizer.save_pretrained("qwen_lora")

## Resources

**Official Unsloth notebooks (reference):**
- [Qwen3 4B Instruct](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)_Instruct.ipynb)
- [Qwen3 4B Thinking](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)_Thinking.ipynb)
- [Gemma 4 E2B Text](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E2B)_Text.ipynb)
- [Qwen3.5 2B Vision](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb)

**Docs:**
- [Fine-tuning Guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide)
- [LoRA Hyperparameters](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)

**What this guide corrects (mistakes I made before):**
- ❌ `assistant_only_loss=True` → ✅ `train_on_responses_only(...)` (Unsloth official)
- ❌ `processing_class=tokenizer` → ✅ `tokenizer=tokenizer` (still used in the official notebooks)
- ❌ `weight_decay=0.01` → ✅ `0.001`
- ❌ `lr_scheduler_type='cosine'` → ✅ `'linear'`
- ❌ `warmup_ratio=0.03` → ✅ `warmup_steps=5`
- ❌ `unsloth/Qwen3.5-2B-unsloth-bnb-4bit` (not on the Hub) → ✅ `unsloth/Qwen3.5-2B`
- ❌ `FastLanguageModel` legacy → ✅ still recommended (for Qwen3)